In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta, timezone

# ----------------------------
# USER SETTINGS
# ----------------------------
WATTTIME_USERNAME=FinnCaseUtil
WATTTIME_PASSWORD=Doughnut12!
REGION = "CAISO_NORTH"   # change if needed
SIGNAL_TYPE = "co2_moer"

MODE = "historical"   # use "historical" for past 24h, "forecast" for next 24h

# ----------------------------
# AUTH
# ----------------------------
auth_resp = requests.get(
    "https://api.watttime.org/login",
    auth=(USERNAME, PASSWORD)
)
auth_resp.raise_for_status()
token = auth_resp.json()["token"]

headers = {
    "Authorization": f"Bearer {token}"
}

# ----------------------------
# TIME WINDOW
# ----------------------------
now_utc = datetime.now(timezone.utc)

if MODE == "historical":
    start = now_utc - timedelta(hours=24)
    end = now_utc
elif MODE == "forecast":
    start = now_utc
    end = now_utc + timedelta(hours=24)
else:
    raise ValueError("MODE must be 'historical' or 'forecast'")

start_str = start.isoformat()
end_str = end.isoformat()

# ----------------------------
# REQUEST DATA
# ----------------------------
if MODE == "historical":
    url = "https://api.watttime.org/v3/historical"
    params = {
        "region": REGION,
        "signal_type": SIGNAL_TYPE,
        "start": start_str,
        "end": end_str,
    }
elif MODE == "forecast":
    url = "https://api.watttime.org/v3/forecast"
    params = {
        "region": REGION,
        "signal_type": SIGNAL_TYPE,
    }

resp = requests.get(url, headers=headers, params=params)
resp.raise_for_status()
data = resp.json()

print("Status:", resp.status_code)
print("First few raw items:", data[:3] if isinstance(data, list) else data)

# ----------------------------
# LOAD TO DATAFRAME
# ----------------------------
# forecast responses often come in a dict with key "data"
if isinstance(data, dict) and "data" in data:
    records = data["data"]
elif isinstance(data, list):
    records = data
else:
    raise ValueError("Unexpected response format")

df = pd.DataFrame(records)

# figure out time/value columns
time_col = "point_time" if "point_time" in df.columns else "timestamp"
value_col = "value"

df[time_col] = pd.to_datetime(df[time_col], utc=True)
df = df.sort_values(time_col)

# for forecast, limit to next 24 hours if endpoint returns more than 24h
df = df[(df[time_col] >= start) & (df[time_col] <= end)].copy()

# convert to local time if you want cleaner charts
df["time_local"] = df[time_col].dt.tz_convert("America/Los_Angeles")
df = df.set_index("time_local")

# ----------------------------
# 5-MINUTE SERIES
# ----------------------------
five_min_df = df[[value_col]].copy()
five_min_df.columns = ["emissions_5min"]

print(five_min_df.head())
print(f"Rows: {len(five_min_df)}")

# ----------------------------
# HOURLY AVERAGE
# ----------------------------
hourly_df = five_min_df.resample("1H").mean().dropna()
hourly_df.columns = ["emissions_hourly_avg"]

print(hourly_df.head())

# ----------------------------
# METRICS
# ----------------------------
daily_avg = hourly_df["emissions_hourly_avg"].mean()

best_6 = hourly_df.nsmallest(6, "emissions_hourly_avg")
worst_6 = hourly_df.nlargest(6, "emissions_hourly_avg")

best_6_avg = best_6["emissions_hourly_avg"].mean()
worst_6_avg = worst_6["emissions_hourly_avg"].mean()

summary_df = pd.DataFrame({
    "category": ["Daily Average", "Best 6 Hours", "Worst 6 Hours"],
    "emissions": [daily_avg, best_6_avg, worst_6_avg]
})

print(summary_df)

HTTPError: 403 Client Error: Forbidden for url: https://api.watttime.org/login